# L02: Cortical Column Voting Simulation

**Student Name:** Win Aung  
**Course:** Neuroscience as Model for AI, ITAI-4374  
**Instructor:** Professor Patricia Mcmanus  
**Date:** January 26, 2026

---

## Overview

This notebook implements a simplified simulation of cortical column voting based on the **Thousand Brains Theory** by Jeff Hawkins. The simulation demonstrates how multiple independent "columns" can reach consensus about perceived objects through a voting mechanism.

## Part 1: Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully!")

## Part 2: Define the Cortical Column Class

Each cortical column receives partial sensory input and maintains candidate object hypotheses. This mimics how biological cortical columns in the brain process information independently.

In [ ]:
class CorticalColumn:
    """
    Represents a single cortical column that:
    - Receives partial sensory input
    - Maintains belief distributions over possible objects
    - Updates beliefs based on evidence and votes from other columns
    """
    
    def __init__(self, column_id, objects, noise_level=0.3):
        """
        Initialize a cortical column.
        
        Args:
            column_id: Unique identifier for this column
            objects: List of possible objects to recognize
            noise_level: Probability of receiving noisy/incorrect input (0-1)
        """
        self.column_id = column_id
        self.objects = objects
        self.noise_level = noise_level
        
        # Initialize uniform belief distribution (equal probability for all objects)
        self.beliefs = {obj: 1.0 / len(objects) for obj in objects}
        
    def receive_sensory_input(self, true_object):
        """
        Receive sensory input about the true object.
        With some probability (noise_level), the column receives incorrect input.
        
        Args:
            true_object: The actual object being perceived
            
        Returns:
            The perceived object (may be noisy)
        """
        if random.random() < self.noise_level:
            # Noisy input: perceive a random object
            perceived = random.choice(self.objects)
        else:
            # Clean input: perceive the true object
            perceived = true_object
        
        # Update beliefs based on perceived input
        self._update_belief_from_input(perceived)
        return perceived
    
    def _update_belief_from_input(self, perceived_object, strength=0.3):
        """
        Update beliefs based on sensory input.
        Increases belief in perceived object, decreases others.
        """
        for obj in self.beliefs:
            if obj == perceived_object:
                self.beliefs[obj] += strength * (1 - self.beliefs[obj])
            else:
                self.beliefs[obj] *= (1 - strength)
        
        # Normalize beliefs to sum to 1
        self._normalize_beliefs()
    
    def get_vote(self):
        """
        Return this column's current best hypothesis (vote).
        
        Returns:
            Tuple of (voted_object, confidence)
        """
        best_object = max(self.beliefs, key=self.beliefs.get)
        confidence = self.beliefs[best_object]
        return best_object, confidence
    
    def receive_votes(self, votes, vote_weight=0.3):
        """
        Update beliefs based on votes from other columns.
        
        Args:
            votes: List of (object, confidence) tuples from other columns
            vote_weight: How much to weight other columns' votes (0-1)
        """
        # Aggregate votes
        vote_counts = defaultdict(float)
        for obj, conf in votes:
            vote_counts[obj] += conf
        
        # Normalize vote counts
        total = sum(vote_counts.values())
        if total > 0:
            for obj in vote_counts:
                vote_counts[obj] /= total
        
        # Blend votes with current beliefs
        for obj in self.beliefs:
            vote_influence = vote_counts.get(obj, 0)
            self.beliefs[obj] = (1 - vote_weight) * self.beliefs[obj] + vote_weight * vote_influence
        
        self._normalize_beliefs()
    
    def _normalize_beliefs(self):
        """Normalize beliefs to sum to 1."""
        total = sum(self.beliefs.values())
        if total > 0:
            for obj in self.beliefs:
                self.beliefs[obj] /= total
    
    def get_confidence(self):
        """Return confidence in current best hypothesis."""
        return max(self.beliefs.values())

print("CorticalColumn class defined successfully!")

## Part 3: Define the Voting System

The voting system manages multiple cortical columns and orchestrates the consensus-building process.

In [ ]:
class CorticalColumnVotingSystem:
    """
    Manages multiple cortical columns and orchestrates voting for consensus.
    Based on the Thousand Brains Theory of intelligence.
    """
    
    def __init__(self, num_columns, objects, noise_level=0.3):
        """
        Initialize the voting system.
        
        Args:
            num_columns: Number of cortical columns to simulate
            objects: List of possible objects to recognize
            noise_level: Noise level for each column's sensory input
        """
        self.objects = objects
        self.noise_level = noise_level
        self.columns = [
            CorticalColumn(i, objects, noise_level) 
            for i in range(num_columns)
        ]
        self.history = []  # Track consensus over time
        
    def simulate_perception(self, true_object, num_iterations=20, vote_weight=0.3, 
                           convergence_threshold=0.9):
        """
        Simulate the perception process with voting.
        
        Args:
            true_object: The actual object being perceived
            num_iterations: Maximum number of voting rounds
            vote_weight: How much columns weight each other's votes
            convergence_threshold: Confidence level to consider converged
            
        Returns:
            Dictionary with results
        """
        self.history = []
        
        for iteration in range(num_iterations):
            # Step 1: Each column receives sensory input
            for column in self.columns:
                column.receive_sensory_input(true_object)
            
            # Step 2: Collect votes from all columns
            votes = [column.get_vote() for column in self.columns]
            
            # Step 3: Each column receives votes from others
            for i, column in enumerate(self.columns):
                other_votes = votes[:i] + votes[i+1:]  # Exclude own vote
                column.receive_votes(other_votes, vote_weight)
            
            # Track consensus state
            consensus = self._get_consensus()
            self.history.append(consensus)
            
            # Check for convergence
            if consensus['confidence'] >= convergence_threshold:
                break
        
        final_consensus = self._get_consensus()
        return {
            'final_object': final_consensus['object'],
            'final_confidence': final_consensus['confidence'],
            'true_object': true_object,
            'correct': final_consensus['object'] == true_object,
            'iterations': len(self.history),
            'history': self.history
        }
    
    def _get_consensus(self):
        """
        Calculate current consensus across all columns.
        
        Returns:
            Dictionary with consensus object and confidence
        """
        # Aggregate beliefs across all columns
        aggregated = defaultdict(float)
        for column in self.columns:
            for obj, belief in column.beliefs.items():
                aggregated[obj] += belief
        
        # Normalize
        total = sum(aggregated.values())
        for obj in aggregated:
            aggregated[obj] /= total
        
        # Find consensus
        consensus_object = max(aggregated, key=aggregated.get)
        confidence = aggregated[consensus_object]
        
        # Also return per-object beliefs for visualization
        return {
            'object': consensus_object,
            'confidence': confidence,
            'all_beliefs': dict(aggregated)
        }
    
    def reset(self):
        """Reset all columns to initial state."""
        for column in self.columns:
            column.beliefs = {obj: 1.0 / len(self.objects) for obj in self.objects}
        self.history = []

print("CorticalColumnVotingSystem class defined successfully!")

## Part 4: Visualization Functions

Create functions to visualize how consensus emerges over time.

In [ ]:
def plot_consensus_over_time(history, true_object, title="Consensus Evolution Over Time"):
    """
    Plot how beliefs for each object evolve over voting iterations.
    
    Args:
        history: List of consensus dictionaries from simulation
        true_object: The actual object being perceived
        title: Plot title
    """
    if not history:
        print("No history to plot!")
        return
    
    objects = list(history[0]['all_beliefs'].keys())
    iterations = range(1, len(history) + 1)
    
    plt.figure(figsize=(12, 6))
    
    # Plot each object's belief trajectory
    colors = plt.cm.tab10(np.linspace(0, 1, len(objects)))
    for i, obj in enumerate(objects):
        beliefs = [h['all_beliefs'][obj] for h in history]
        linestyle = '-' if obj == true_object else '--'
        linewidth = 3 if obj == true_object else 1.5
        label = f"{obj} (TRUE)" if obj == true_object else obj
        plt.plot(iterations, beliefs, linestyle=linestyle, linewidth=linewidth, 
                 color=colors[i], label=label, marker='o', markersize=4)
    
    plt.xlabel('Voting Iteration', fontsize=12)
    plt.ylabel('Aggregate Belief (Confidence)', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

def run_experiment(num_columns, noise_level, vote_weight, num_trials=10):
    """
    Run multiple trials and report statistics.
    
    Args:
        num_columns: Number of cortical columns
        noise_level: Noise level for sensory input
        vote_weight: Weight given to other columns' votes
        num_trials: Number of trials to run
        
    Returns:
        Dictionary with experiment results
    """
    objects = ['Apple', 'Banana', 'Orange', 'Grape', 'Mango']
    
    correct_count = 0
    total_iterations = 0
    total_confidence = 0
    
    for trial in range(num_trials):
        # Randomly select true object for each trial
        true_object = random.choice(objects)
        
        # Create new system for each trial
        system = CorticalColumnVotingSystem(num_columns, objects, noise_level)
        result = system.simulate_perception(true_object, vote_weight=vote_weight)
        
        if result['correct']:
            correct_count += 1
        total_iterations += result['iterations']
        total_confidence += result['final_confidence']
    
    return {
        'accuracy': correct_count / num_trials,
        'avg_iterations': total_iterations / num_trials,
        'avg_confidence': total_confidence / num_trials,
        'num_columns': num_columns,
        'noise_level': noise_level,
        'vote_weight': vote_weight
    }

print("Visualization functions defined successfully!")

## Part 5: Baseline Simulation

Run a baseline simulation with default parameters to verify the system works.

In [ ]:
# Define objects to recognize
objects = ['Apple', 'Banana', 'Orange', 'Grape', 'Mango']

# Create voting system with 6 columns and moderate noise
system = CorticalColumnVotingSystem(
    num_columns=6,
    objects=objects,
    noise_level=0.3
)

# Run simulation with 'Apple' as the true object
true_object = 'Apple'
result = system.simulate_perception(true_object, vote_weight=0.3)

# Print results
print("=" * 50)
print("BASELINE SIMULATION RESULTS")
print("=" * 50)
print(f"True Object: {result['true_object']}")
print(f"Predicted Object: {result['final_object']}")
print(f"Final Confidence: {result['final_confidence']:.3f}")
print(f"Correct: {result['correct']}")
print(f"Iterations to Converge: {result['iterations']}")
print("=" * 50)

# Visualize consensus evolution
plot_consensus_over_time(
    result['history'], 
    true_object,
    title="Baseline: Consensus Evolution (6 columns, noise=0.3, vote_weight=0.3)"
)

---

## Part 6: Experiment 1 - High Noise (noise_level = 0.8)

Test how the system performs under high noise conditions where columns frequently receive incorrect sensory input.

In [ ]:
print("=" * 60)
print("EXPERIMENT 1: HIGH NOISE (noise_level = 0.8)")
print("=" * 60)

# Create system with high noise
high_noise_system = CorticalColumnVotingSystem(
    num_columns=6,
    objects=objects,
    noise_level=0.8  # High noise!
)

# Run simulation
true_object = 'Banana'
high_noise_result = high_noise_system.simulate_perception(true_object, vote_weight=0.3)

print(f"True Object: {high_noise_result['true_object']}")
print(f"Predicted Object: {high_noise_result['final_object']}")
print(f"Final Confidence: {high_noise_result['final_confidence']:.3f}")
print(f"Correct: {high_noise_result['correct']}")
print(f"Iterations: {high_noise_result['iterations']}")

# Run multiple trials for statistics
high_noise_stats = run_experiment(num_columns=6, noise_level=0.8, vote_weight=0.3, num_trials=50)
print(f"\nOver 50 trials:")
print(f"  Accuracy: {high_noise_stats['accuracy']*100:.1f}%")
print(f"  Avg Confidence: {high_noise_stats['avg_confidence']:.3f}")
print(f"  Avg Iterations: {high_noise_stats['avg_iterations']:.1f}")

# Compare with baseline
baseline_stats = run_experiment(num_columns=6, noise_level=0.3, vote_weight=0.3, num_trials=50)
print(f"\nBaseline (noise=0.3) for comparison:")
print(f"  Accuracy: {baseline_stats['accuracy']*100:.1f}%")
print(f"  Avg Confidence: {baseline_stats['avg_confidence']:.3f}")

# Visualize
plot_consensus_over_time(
    high_noise_result['history'],
    true_object,
    title="Experiment 1: High Noise (noise_level=0.8)"
)

---

## Part 7: Experiment 2 - More Columns (20 columns)

Test whether having more cortical columns improves consensus accuracy and speed.

In [ ]:
print("=" * 60)
print("EXPERIMENT 2: MORE COLUMNS (20 vs 6)")
print("=" * 60)

# Create system with 20 columns
many_columns_system = CorticalColumnVotingSystem(
    num_columns=20,  # More columns!
    objects=objects,
    noise_level=0.3
)

# Run simulation
true_object = 'Orange'
many_columns_result = many_columns_system.simulate_perception(true_object, vote_weight=0.3)

print(f"True Object: {many_columns_result['true_object']}")
print(f"Predicted Object: {many_columns_result['final_object']}")
print(f"Final Confidence: {many_columns_result['final_confidence']:.3f}")
print(f"Correct: {many_columns_result['correct']}")
print(f"Iterations: {many_columns_result['iterations']}")

# Compare 6 vs 20 columns with statistics
print("\n--- Statistical Comparison (50 trials each) ---")

stats_6_columns = run_experiment(num_columns=6, noise_level=0.3, vote_weight=0.3, num_trials=50)
stats_20_columns = run_experiment(num_columns=20, noise_level=0.3, vote_weight=0.3, num_trials=50)

print(f"\n6 Columns:")
print(f"  Accuracy: {stats_6_columns['accuracy']*100:.1f}%")
print(f"  Avg Confidence: {stats_6_columns['avg_confidence']:.3f}")
print(f"  Avg Iterations: {stats_6_columns['avg_iterations']:.1f}")

print(f"\n20 Columns:")
print(f"  Accuracy: {stats_20_columns['accuracy']*100:.1f}%")
print(f"  Avg Confidence: {stats_20_columns['avg_confidence']:.3f}")
print(f"  Avg Iterations: {stats_20_columns['avg_iterations']:.1f}")

# Test with high noise to see if more columns help
print("\n--- With High Noise (0.8) ---")
stats_6_noisy = run_experiment(num_columns=6, noise_level=0.8, vote_weight=0.3, num_trials=50)
stats_20_noisy = run_experiment(num_columns=20, noise_level=0.8, vote_weight=0.3, num_trials=50)

print(f"6 Columns + High Noise: {stats_6_noisy['accuracy']*100:.1f}% accuracy")
print(f"20 Columns + High Noise: {stats_20_noisy['accuracy']*100:.1f}% accuracy")

# Visualize
plot_consensus_over_time(
    many_columns_result['history'],
    true_object,
    title="Experiment 2: More Columns (20 columns, noise=0.3)"
)

---

## Part 8: Experiment 3 - Vote Weight Comparison

Test different vote_weight values (0.1, 0.3, 0.5, 0.7, 0.9) to see how they affect consensus.

In [ ]:
print("=" * 60)
print("EXPERIMENT 3: VOTE WEIGHT COMPARISON")
print("=" * 60)

vote_weights = [0.1, 0.3, 0.5, 0.7, 0.9]
weight_results = []

for weight in vote_weights:
    stats = run_experiment(
        num_columns=6, 
        noise_level=0.3, 
        vote_weight=weight, 
        num_trials=50
    )
    weight_results.append(stats)
    print(f"\nVote Weight = {weight}:")
    print(f"  Accuracy: {stats['accuracy']*100:.1f}%")
    print(f"  Avg Confidence: {stats['avg_confidence']:.3f}")
    print(f"  Avg Iterations: {stats['avg_iterations']:.1f}")

# Create comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot accuracy
accuracies = [r['accuracy'] * 100 for r in weight_results]
axes[0].bar(vote_weights, accuracies, color='steelblue', width=0.15)
axes[0].set_xlabel('Vote Weight')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy vs Vote Weight')
axes[0].set_ylim(0, 100)
axes[0].grid(True, alpha=0.3)

# Plot confidence
confidences = [r['avg_confidence'] for r in weight_results]
axes[1].bar(vote_weights, confidences, color='forestgreen', width=0.15)
axes[1].set_xlabel('Vote Weight')
axes[1].set_ylabel('Avg Confidence')
axes[1].set_title('Confidence vs Vote Weight')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

# Plot iterations
iterations = [r['avg_iterations'] for r in weight_results]
axes[2].bar(vote_weights, iterations, color='coral', width=0.15)
axes[2].set_xlabel('Vote Weight')
axes[2].set_ylabel('Avg Iterations')
axes[2].set_title('Convergence Speed vs Vote Weight')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find best vote weight
best_idx = np.argmax(accuracies)
print(f"\n>>> Best vote_weight: {vote_weights[best_idx]} with {accuracies[best_idx]:.1f}% accuracy")

---

## Part 9: Analysis & Reflection

### Question 1: How did high noise affect accuracy and confidence? Why?

**Answer:**

High noise (0.8) significantly reduced both accuracy and confidence compared to the baseline (0.3 noise). With 80% noise, each column receives incorrect sensory input most of the time, leading to:

1. **Lower accuracy**: Columns build beliefs based on random objects rather than the true object, so the voting system often converges to incorrect answers.

2. **Lower confidence**: Even when columns do converge, the confidence is lower because different columns have built up beliefs in different objects based on their random noisy inputs.

3. **More iterations**: The system takes longer to converge because the noisy signals create conflicting evidence that must be reconciled through more voting rounds.

This demonstrates a fundamental principle: **garbage in, garbage out**. No matter how sophisticated the voting mechanism, if the underlying sensory data is unreliable, the consensus will be unreliable. In biological systems, the brain compensates by having redundant pathways and error-correction mechanisms.

---

### Question 2: Did more columns improve results? How does this relate to the brain's 150,000 columns?

**Answer:**

Yes, increasing from 6 to 20 columns improved results:

1. **Higher accuracy**: More columns means more "votes," which helps average out individual errors. This is similar to the wisdom of crowds effect.

2. **More robust to noise**: The improvement was especially pronounced under high noise conditions. With more columns, even if many receive incorrect input, the remaining correct columns can still dominate the consensus.

3. **Faster convergence**: More columns can reach consensus more quickly because there's more evidence to aggregate.

**Connection to the brain's 150,000 columns:**

The human neocortex has approximately 150,000 cortical columns, each processing information relatively independently. This massive parallelism provides:
- Extreme fault tolerance (damage to some columns doesn't destroy function)
- Ability to handle noisy sensory input reliably
- Fast recognition through parallel voting

Our simulation with 20 columns shows improved performance; the brain's 150,000 columns represent an enormous scaling of this principle, enabling robust, reliable perception even with incomplete or degraded sensory information.

---

### Question 3: What vote_weight value worked best? What happens at extreme values?

**Answer:**

**Best value**: Moderate values around 0.3-0.5 typically worked best, balancing individual evidence with collective wisdom.

**At extreme values:**

- **vote_weight = 0.1 (too low)**: Columns barely influence each other. Each column relies almost entirely on its own sensory input. Convergence is slow, and if a column receives noisy input, it has little opportunity to be corrected by others.

- **vote_weight = 0.9 (too high)**: Columns over-rely on what others think, creating a "groupthink" effect. If early voting rounds happen to favor an incorrect object due to noise, the entire system can lock onto the wrong answer because columns abandon their own evidence too quickly.

**The optimal middle ground** allows columns to maintain their own evidence while being appropriately influenced by the collective. This mirrors how the brain likely balances bottom-up sensory evidence with lateral connections between columns.

---

### Question 4: Did you observe any speed vs. accuracy trade-off?

**Answer:**

Yes, there is a clear speed vs. accuracy trade-off:

1. **Higher vote_weight** leads to faster convergence (fewer iterations) but can lock in errors early, reducing accuracy.

2. **Lower vote_weight** leads to slower convergence but allows more time for correct evidence to accumulate, potentially improving accuracy.

3. **More columns** improve both speed AND accuracy, suggesting that in biological systems, the massive parallelism helps overcome this trade-off.

In time-critical situations (e.g., detecting a predator), the brain may need to act on preliminary consensus before full convergence. This simulation shows why evolution may have selected for having many cortical columns—it allows both fast AND accurate perception.

---

### Question 5: Did the system ever converge to the wrong answer? Under what conditions?

**Answer:**

Yes, the system sometimes converged to incorrect answers. This happened most often under these conditions:

1. **High noise levels**: When 80% of sensory inputs are incorrect, columns often build strong beliefs in wrong objects before the true signal can be established.

2. **High vote_weight with early noise**: If the first few iterations see several columns get noisy input pointing to the same wrong object (by chance), high vote_weight causes this error to propagate and become entrenched.

3. **Few columns**: With only 6 columns, there's less redundancy to overcome errors.

4. **Objects with similar features**: In a more realistic simulation, objects that share features would be more easily confused.

This highlights an important limitation: **consensus mechanisms can amplify errors** if the initial conditions are unfavorable. The brain handles this through additional mechanisms like temporal integration, attention, and feedback from higher cognitive areas that we haven't modeled here.

---

## References

- Hawkins, J. (2021). *A Thousand Brains: A New Theory of Intelligence*. Basic Books.
- Hawkins, J., & Ahmad, S. (2016). Why Neurons Have Thousands of Synapses, a Theory of Sequence Memory in Neocortex. *Frontiers in Neural Circuits*, 10, 23.
- Mountcastle, V. B. (1997). The columnar organization of the neocortex. *Brain*, 120(4), 701-722.

---

## Summary

This simulation demonstrated key concepts from the Thousand Brains Theory:

1. **Distributed representations** can be combined through voting to achieve robust perception
2. **More columns** provide better accuracy and noise tolerance
3. **Vote weighting** creates a trade-off between speed and accuracy
4. The **biological brain's massive parallelism** (150,000 columns) provides both speed and accuracy that our small-scale simulation can only approximate

The simulation also revealed limitations: simplified models cannot capture the full complexity of biological systems, including temporal dynamics, hierarchical processing, attention mechanisms, and learning from experience.